## Geographic heatmap - a 'choropleth map' of the US.
### Goal: visualize geographic data to gain regional insight
### Step 1: Imports

In [7]:
import requests
import pandas as pd
import plotly.express as px # provides the plotting
from io import StringIO # converts html into a 'StringIO object'

# List of metrics supported by the demo.
METRICS = ['incarceration_and_correctional_supervision_rate', 'income', "poverty_rate"]

# Select specific metric for the analysis
METRIC = METRICS[1]

In [ ]:
# Define data_lookup -- description of the 'region' column, the 'metric' column and a text description of the metric. This is based on the METRIC used for the analysis
data_lookup_master = {"poverty_rate": {'region' : 'Area', 'metric':'Percent', 'metric_name':'Poverty rate'},
               "income" : {'region': 'States and D.C.', 'metric':'2023', 'metric_name':'Median income'},
                "incarceration_and_correctional_supervision_rate": {'region': 'Location', 'metric' : 'Incarceration rate', 'metric_name':'Incarceration rate'}
}
data_lookup = data_lookup_master[METRIC]


region_col = data_lookup['region']
metric_col = data_lookup['metric']
metric_name = data_lookup['metric_name']

### Step 2: Demographic data from wikipedia

In [9]:
url = "https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_"+METRIC

# Wikipedia sometimes 403s (forbidden) Python's default user agent, so pretend to be a browser
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()  # nice clear error if something goes wrong

html = response.text

# Quick sanity check: how many tables did we find?
# Note: Table structure is correct as of Nov 2025 and may change later.
tables = pd.read_html(StringIO(html))
print(f"# of tables found: {len(tables)}")

# Find the right table by verifying the region_col and metric_col are in it.
target = None
for t in tables:
    cols = [str(c) for c in t.columns]
    if region_col in cols and metric_col in cols:
        target = t
        break
if target is None:
    raise RuntimeError("Could not find the table. Wikipedia layout may have changed.")


# of tables found: 9


In [10]:
# put found table in wiki_df
wiki_df = target.copy()

wiki_df.head()

,States and D.C.,2023,2022,2021,2019,2018,2017,2016,2015,2014,2013,Growth rate
0,United States,"$77,719","$74,755","$69,717","$65,712","$63,179","$60,336","$57,617","$55,775","$53,657","$52,250",3.07%
1,"Washington, D.C.","$108,210","$101,027","$90,088","$92,266","$85,203","$82,372","$75,506","$75,628","$71,648","$67,572",4.82%
2,Massachusetts,"$99,858","$94,488","$89,645","$85,843","$79,835","$77,385","$75,297","$70,628","$69,160","$66,768",4.11%
3,New Jersey,"$99,781","$96,346","$89,296","$85,751","$81,740","$80,088","$76,126","$72,222","$71,919","$70,165",3.58%
4,Maryland,"$98,678","$94,991","$90,203","$86,738","$83,242","$80,776","$78,945","$75,847","$73,971","$72,483",3.13%


### Step 3: Perform data cleaning
<ul>
<li>Convert region_col into a state's 2-letter code</li>
<li>Convert metric_col into a number</li>
</ul>

In [11]:
# Map full state names to 2-letter state codes for Plotly
    # Clean up state name -- some have brackets and such
wiki_df = wiki_df.rename(columns={region_col: "state_name"}) # rename first column to 'state_name'
wiki_df["state_name"] = (wiki_df["state_name"] # select the column
                         .astype(str)          # make sure it's a string
                        .str.replace(r"^\s+", "", regex=True) 
                         .str.replace(r"\[.*\]", "", regex=True) # standardize state name -- some have brackets.
                         .str.replace(r"\s\*","", regex=True)
)

    # Define mapping to 2-letter code
state_to_code = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
    "Florida": "FL", "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID",
    "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
    "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
    "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
    "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM",
    "New York": "NY", "North Carolina": "NC", "North Dakota": "ND",
    "Ohio": "OH", "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA",
    "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
    "Wisconsin": "WI", "Wyoming": "WY",
    "District of Columbia": "DC",
}
# Keep only actual states + DC (drop territories with no code mapped)
wiki_df["state_code"] = [state_to_code[x] if x in state_to_code.keys() else None for x in wiki_df["state_name"]]
wiki_df = wiki_df.dropna(subset=["state_code"]).copy()[['state_name','state_code', metric_col]]


# Convert the
wiki_df[metric_name] = (
    wiki_df[str(metric_col)]                  # grab the most recent year
    .astype(str)                          # make sure the type is string
    .str.replace(r"[$,]*","", regex=True)
    .astype(float)                        # convert to float
)

wiki_df = wiki_df[['state_code','state_name', metric_name]]
wiki_df.head()

,state_code,state_name,Median income
2,MA,Massachusetts,99858.0
3,NJ,New Jersey,99781.0
4,MD,Maryland,98678.0
5,NH,New Hampshire,96838.0
6,CA,California,95521.0


### Plot results

In [12]:
fig = px.choropleth(
    wiki_df,
    locations="state_code",
    locationmode="USA-states",
    color=metric_name,
    scope="usa",
    color_continuous_scale="Viridis",
    labels={metric_name: metric_name},
    hover_name="state_name",
)

fig.update_layout(title=f"US {metric_name}, Source: Wikipedia")
fig.show()
